---
# **Práctica 3**: *Representaciones Vectoriales*
---
### **Alumno**:  Roberto Samuel Sanchez Rosas

## **Matrices dispersas y búsqueda de documentos**

Este apartado requiere que construyas un motor de búsqueda entre documentos comparando el rendimiento de una Bolsa de Palabras (BoW) y TF-IDF para procesar un documento "tramposo" (documento con muchas palabras que aportan poco significado o valor temático):

1. Define una lista de 5 documentos cortos divididos en dos temas contrastantes.
    - Ej: 3 de Revolución Rusa y 2 de comida vegana.
2. Crea una query "tramposa", esto es, crea un documento dirigido a alguna temática pero repitiendo intencionalmente palabras comunes o verbos genéricos que aparezcan en los documentos de la otra temática.
3. Vectoriza para crear una BoW y calcula la similitud coseno entre la query y los 5 documentos
4. Repite el proceso usando TF-IDF
5. Imprime un DataFrame o tabla comparativa que muestre los scores de similitud de BoW y TF-IDF del query contra los 5 documentos.
    - ¿Cambió el documento clasificado como "más similar/relevante" al pasar de BoW a TF-IDF? Identifica el cambio si lo hubo.
    - Explica brevemente, basándote en la penalización idf (Inverse Document Frequency), cómo y por qué TF-IDF procesó de manera distinta las palabras de tu "trampa léxica" en comparación con BoW.

###**1. Defición 5 documentos**

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Tema A: Backend
doc_back_1 = "El desarrollo del nuevo sistema garantiza la seguridad de los datos."
doc_back_2 = "Un sistema con buena seguridad mejora el desarrollo y protege los datos."
doc_back_3 = "La seguridad en el desarrollo de cualquier sistema de datos es fundamental."

# Tema B: Adopción de Mascotas
doc_pets_1 = "El refugio busca promover la adopción responsable de cada perro rescatado."
doc_pets_2 = "Dale un hogar amoroso a una mascota; la adopción cambia la vida de los animales en el refugio."

documentos = [doc_back_1, doc_back_2, doc_back_3, doc_pets_1, doc_pets_2]
titulos = ["BACKEND_1", "BACKEND_2", "BACKEND_3", "MASCOTAS_1", "MASCOTAS_2"]

###**2. Query**

In [2]:
# Query "tramposa", objetivo real Adopción de Mascotas.
query_trampa = "El desarrollo y la seguridad de los datos ayudan a mejorar la adopción de perros en el refugio. La supervisión y seguridad adecuadas facilitan la adopción de los animales."

documentos.append(query_trampa)
titulos.append("QUERY")

###**3 y 4. Generación de matrices**

In [3]:
import re
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

def simple_preprocess(text: str):
    tokens = word_tokenize(text.lower(), language="spanish")
    # Ignoramos signos de puntuación y palabras de longitud 1
    return [word for word in tokens if word.isalnum() and len(word) > 1 and not re.match(r"\d+", word)]

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
def create_bow_dataframe(docs_raw: list, titles: list[str], vectorizer) -> pd.DataFrame:
    # fit_transform ajusta el vocabulario y crea la matriz en un solo paso
    matrix = vectorizer.fit_transform(docs_raw)

    # Podemos crear el DataFrame directamente pasando la matriz a un array tradicional
    # vectorizer.get_feature_names_out() nos da la lista de palabras en el orden exacto de las columnas
    df = pd.DataFrame(
        matrix.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
    )
    return df

In [5]:
matriz_bow = create_bow_dataframe(documentos, titulos, vectorizer=CountVectorizer(tokenizer=simple_preprocess, token_pattern=None))
matriz_tfidf = create_bow_dataframe(documentos, titulos, vectorizer=TfidfVectorizer(tokenizer=simple_preprocess, token_pattern=None))

# Extraemos los vectores de la query
query_vector_bow = matriz_bow.loc["QUERY"].values.reshape(1, -1)
query_vector_tfidf = matriz_tfidf.loc["QUERY"].values.reshape(1, -1)

# Calculamos similitudes iterando sobre los 5 documentos
resultados = []

for doc_title in titulos[:-1]:
    # Similitud BoW
    doc_vec_bow = matriz_bow.loc[doc_title].values.reshape(1, -1)
    sim_bow = cosine_similarity(doc_vec_bow, query_vector_bow)[0][0]

    # Similitud TF-IDF
    doc_vec_tfidf = matriz_tfidf.loc[doc_title].values.reshape(1, -1)
    sim_tfidf = cosine_similarity(doc_vec_tfidf, query_vector_tfidf)[0][0]

    resultados.append({"Documento": doc_title, "Similitud BoW": sim_bow, "Similitud TF-IDF": sim_tfidf})

###**5. Imprimir un DataFrame con los scores**

In [6]:
df_comparativo = pd.DataFrame(resultados)
print(df_comparativo.sort_values(by="Similitud TF-IDF", ascending=False).to_markdown(index=False))

| Documento   |   Similitud BoW |   Similitud TF-IDF |
|:------------|----------------:|-------------------:|
| MASCOTAS_2  |        0.636285 |           0.426411 |
| BACKEND_3   |        0.630062 |           0.425078 |
| BACKEND_1   |        0.627182 |           0.40584  |
| MASCOTAS_1  |        0.501745 |           0.293011 |
| BACKEND_2   |        0.334497 |           0.213345 |


#### ¿Cambió el documento clasificado como "más similar/relevante" al pasar de BoW a TF-IDF?
No cambió el documento clasificado como más similar, en ambos casos, el documento MASCOTAS_2 sigue siendo el más relevante. Sin embargo, las puntuaciones de los documentos del tema backend disminuyeron ligeramente al usar TF-IDF, mientras que las palabras clave del tema real (adopción de mascotas) ganaron peso relativo.

Esto ocurre porque TF-IDF penaliza las palabras muy frecuentes en todos los documentos mediante la componente IDF (Inverse Document Frequency). Las palabras “trampa” de backend como desarrollo, seguridad o datos aparecen en muchos documentos y, por tanto, tienen un IDF bajo, reduciendo su influencia en la similitud. En cambio, palabras menos frecuentes y más representativas del tema real (adopción, perro, refugio) reciben mayor peso, haciendo que TF-IDF capture mejor la intención del query que BoW, que solo cuenta repeticiones sin ponderar importancia.

## Busqueda de sesgos

1. Descarga el modelo `glove-wiki-gigaword-100` con la api de `gensim` y ejecuta el siguiente código:

```python
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))
```

2. Identifica las diferencias en la lista de palabras asociadas a hombres/mujeres y profesiones, explica como esto reflejaría un sesgo de genero.
3. Utiliza la función `.most_similar()` para identificar analogías que exhiba algún tipo de sesgo de los vectores pre-entrenados.
    - Explica brevemente que sesgo identificar
4. Si fuera tu trabajo crear un modelo ¿Como mitigarías estos sesgos al crear vectores de palabras?

In [7]:
pip install gensim

###**1. Descargar glove-wiki-gigaword-100**

In [8]:
import gensim.downloader as api

# Descargamos el modelo GloVe entrenado en Wikipedia y Gigaword
word_vectors = api.load("glove-wiki-gigaword-100")

print("Hombre a profesión:")
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print("\nMujer a profesión:")
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))

Hombre a profesión:
[('practice', 0.6156836152076721), ('knowledge', 0.6129590272903442), ('teaching', 0.5949095487594604), ('skill', 0.5886170864105225), ('reputation', 0.588079571723938), ('philosophy', 0.5868663191795349), ('work', 0.5848589539527893), ('skills', 0.5772278904914856), ('discipline', 0.576593816280365), ('mind', 0.5739315152168274)]

Mujer a profesión:
[('professions', 0.6473134756088257), ('practitioner', 0.5966603755950928), ('nursing', 0.5942842364311218), ('vocation', 0.5698666572570801), ('teaching', 0.5623518824577332), ('childbirth', 0.543552815914154), ('academic', 0.5408717393875122), ('teacher', 0.5401058197021484), ('educator', 0.5207306742668152), ('qualifications', 0.5143449902534485)]


###**2. Identificación de las diferencias**

En los resultados para hombre, la palabra profesión se asocia con autoridad, intelecto o áreas técnicas. Para mujer, aparece ligada a cuidado, educación o parto. Esto muestra un sesgo de género en los embeddings porque el modelo refleja patrones de los textos con los que se entrenó y no la realidad. Palabras como childbirth destacan que el modelo relaciona automáticamente características biológicas de las mujeres con su rol profesional.

###**3. Función .most_similar()**

In [9]:
# Identificación de sesgo en áreas tecnológicas
print("Hombre es a Programador como Mujer es a:")
print(word_vectors.most_similar(positive=['programmer', 'woman'], negative=['man']))

Hombre es a Programador como Mujer es a:
[('educator', 0.5853328704833984), ('programmers', 0.5730530023574829), ('linguist', 0.5432007312774658), ('technician', 0.5373360514640808), ('freelance', 0.5362123250961304), ('animator', 0.5339367389678955), ('translator', 0.533419132232666), ('software', 0.4966624081134796), ('psychotherapist', 0.49566227197647095), ('technologist', 0.4880514144897461)]


Al pedir la contraparte femenina de "programador", el modelo devuelve primero "educator" y luego profesiones relacionadas con enseñanza, comunicación o cuidado. Esto muestra un sesgo de género porque no asocia a la mujer con la ingeniería o el desarrollo de software, sino con roles históricamente considerados femeninos. Los vectores reflejan los patrones de los textos con los que se entrenó el modelo y no la equivalencia real de los roles.

###**¿Como mitigar estos sesgos?**

Para reducir los sesgos al crear vectores de palabras, primero entrenaría el modelo con textos más diversos y equilibrados, que incluyan distintos géneros, culturas y roles laborales, para que no refleje solo estereotipos históricos. También aplicaría métodos para ajustar los vectores, de manera que las palabras de género no determinen automáticamente la asociación con profesiones u otras características. Por último, revisaría los resultados antes de usar el modelo para detectar posibles asociaciones injustas y corregirlas.